# FabricFlow Component 3 — Model Explanation

**Student ID:** IT22904614  
**Research component:** Emergency Situation Detection and Management

This notebook explains the complete Component 3 machine-learning workflow using the project's real code, datasets, saved artifacts, and evaluation reports. It is designed for thesis evidence, final-presentation demonstrations, and viva questions.

> Research boundary: the saved models are working academic-prototype artifacts. They are not presented as production-approved models.

## How to run this notebook

### Local VS Code / Jupyter

Open the project root in VS Code, select the project's `.venv` Python interpreter, and run the cells from top to bottom. If notebook support or charts are unavailable, install them without changing the application runtime:

```bash
source .venv/bin/activate
python -m pip install ipykernel matplotlib
```

### Google Colab

Upload/extract the full project folder to Colab or mount Google Drive, change into the project directory, and then run the cells. Colab is optional; it is only an execution environment.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/Garment_model_backend_latest"),
]
PROJECT_ROOT = next(
    (
        path.resolve()
        for path in ROOT_CANDIDATES
        if (path / "components" / "component3.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Run this notebook from the FabricFlow project or its notebooks directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

## 1. What Component 3 predicts

Component 3 contains several related prediction tasks rather than one single model:

1. **Current risk type:** classifies today's production situation into seven operational classes.
2. **Overall order risk:** estimates whether the remaining order has high delivery exposure.
3. **Three-production-day warnings:** predicts machine breakdown, quality-limit, and output/schedule risks separately.
4. **Recovery planning:** converts the detected state into feasible actions using an explainable deterministic rule engine; this part is not an ML model.

Runtime flow:

```text
C2 assigned order → daily production data → feature engineering
→ current-risk model + order-risk model → early-warning models
→ calibrated thresholds → recovery rules → monitoring/recovery history → C4
```

In [ ]:
from components.component3_historical_validation import load_historical_workbook

DATASET_PATH = PROJECT_ROOT / "data" / "component3_final_preprocessed_dataset.xlsx"
raw_data, workbook_report = load_historical_workbook(DATASET_PATH)

print("Dataset:", DATASET_PATH.name)
print("Rows:", len(raw_data))
print("Columns:", len(raw_data.columns))
print("Bulk orders:", raw_data["Bulk_Order_ID"].nunique())
display(raw_data.head(5))

## 2. Input parameters and feature engineering

The models use operational measurements that are available on the monitoring day. Buyer name, plant identity, future outcomes, and actual completion dates are not used as predictive inputs. This reduces identity memorisation and target leakage.

In [ ]:
from components.component3_early_warning_data import EARLY_WARNING_FEATURES, TRAILING_FEATURES
from components.component3_features import FEATURES
from components.component3_order_risk_features import DEADLINE_FEATURES, ORDER_RISK_FEATURES

feature_summary = pd.DataFrame(
    [
        {
            "Model input": "Current risk type",
            "Feature count": len(FEATURES),
            "Additional evidence": "Current-day production and disruption fields",
        },
        {
            "Model input": "Overall order risk v3",
            "Feature count": len(ORDER_RISK_FEATURES),
            "Additional evidence": ", ".join(DEADLINE_FEATURES),
        },
        {
            "Model input": "Three-day early warnings",
            "Feature count": len(EARLY_WARNING_FEATURES),
            "Additional evidence": ", ".join(TRAILING_FEATURES),
        },
    ]
)
display(feature_summary)

print("Canonical current-day features:")
display(pd.DataFrame({"Feature": FEATURES}))

In [ ]:
from components.component3_early_warning_data import run_step5a_audit

step5a_report, early_warning_data = run_step5a_audit(DATASET_PATH)
targets = [
    "Machine_Breakdown_Within_3_Days",
    "Quality_Limit_Within_3_Days",
    "Output_Schedule_Risk_Within_3_Days",
]
coverage_rows = []
for target in targets:
    positive = early_warning_data[target].astype(int).eq(1)
    coverage_rows.append(
        {
            "Target": target,
            "Eligible rows": len(early_warning_data),
            "Positive rows": int(positive.sum()),
            "Negative rows": int((~positive).sum()),
            "Positive orders": int(early_warning_data.loc[positive, "Bulk_Order_ID"].nunique()),
            "Negative orders": int(early_warning_data.loc[~positive, "Bulk_Order_ID"].nunique()),
        }
    )
display(pd.DataFrame(coverage_rows))
print("Early-warning eligible orders:", early_warning_data["Bulk_Order_ID"].nunique())

## 3. Why these algorithms were considered

| Algorithm | Research role | Reason |
|---|---|---|
| Dummy classifier | Non-selectable baseline | Demonstrates whether a trained model beats class frequency |
| Logistic Regression | Regularised linear candidate | Stable on a small dataset, class-weighted, interpretable, and probability-friendly |
| Random Forest | Nonlinear ensemble candidate | Captures feature interactions and class imbalance without feature scaling |
| Extra Trees | Alternative tree ensemble | Tests whether stronger tree randomisation improves current-risk classification |
| Gradient Boosting | Sequential ensemble candidate | Tests boosted nonlinear learning for overall order risk |
| XGBoost | Boosted early-warning candidate | Provides a stronger nonlinear comparison, but is selected only when held-out evidence improves |
| Soft Voting | Explicit combined ensemble | Tests whether probability averaging between tuned linear and nonlinear models improves generalisation |

**Selection priority:** unseen-order Macro-F1, then positive-class F1, then Accuracy. Macro-F1 is prioritised because emergency targets are imbalanced.

In [ ]:
artifact_paths = {
    "Current risk type": PROJECT_ROOT / "models" / "c3_model1_risk_type_v2.pkl",
    "Overall order risk v3": PROJECT_ROOT / "models" / "c3_model2_order_risk_v3.pkl",
    "Machine warning": PROJECT_ROOT / "models" / "c3_early_warning_machine_breakdown_v2.joblib",
    "Quality warning": PROJECT_ROOT / "models" / "c3_early_warning_quality_limit_v2.joblib",
    "Output/schedule warning": PROJECT_ROOT
    / "models"
    / "c3_early_warning_output_schedule_risk_v2.joblib",
}

artifact_rows = []
loaded_artifacts = {}
for purpose, path in artifact_paths.items():
    artifact = joblib.load(path)
    loaded_artifacts[purpose] = artifact
    if isinstance(artifact, dict):
        estimator = artifact.get("estimator")
        model_name = artifact.get("model_name", type(estimator).__name__)
        threshold = artifact.get("decision_threshold")
    else:
        estimator = artifact
        model_name = type(estimator).__name__
        threshold = None
    artifact_rows.append(
        {
            "Purpose": purpose,
            "Artifact": path.name,
            "Saved model": model_name,
            "Wrapper": type(estimator).__name__,
            "Decision threshold": threshold,
        }
    )

display(pd.DataFrame(artifact_rows))

## 4. Current-risk and overall-order-risk evidence

Complete bulk orders are held out during validation. Augmented rows are allowed only in training folds and are excluded from validation evidence.

In [ ]:
v2_report = json.loads((PROJECT_ROOT / "reports" / "component3_v2" / "evaluation.json").read_text())
v3_report = json.loads(
    (PROJECT_ROOT / "reports" / "component3_model2_v3" / "evaluation.json").read_text()
)

core_rows = []
for candidate, metrics in v2_report["all_candidates"]["risk_type"].items():
    core_rows.append(
        {
            "Task": "Current risk type",
            "Candidate": candidate,
            "Accuracy": metrics["accuracy"],
            "Macro-F1": metrics["macro_f1"],
            "Weighted-F1": metrics["weighted_f1"],
            "Selected": candidate == "random_forest",
        }
    )
for candidate, metrics in v3_report["all_candidates"].items():
    core_rows.append(
        {
            "Task": "Overall order risk v3",
            "Candidate": candidate,
            "Accuracy": metrics["accuracy"],
            "Macro-F1": metrics["macro_f1"],
            "Weighted-F1": metrics["weighted_f1"],
            "Selected": candidate == v3_report["metadata"]["selected_model"],
        }
    )
core_results = pd.DataFrame(core_rows)
display(
    core_results.style.format({"Accuracy": "{:.2%}", "Macro-F1": "{:.2%}", "Weighted-F1": "{:.2%}"})
)

In [ ]:
risk_labels = list(v2_report["metadata"]["risk_type_mapping"])
risk_matrix = v2_report["metadata"]["selected_metrics"]["risk_type"]["confusion_matrix"]
order_matrix = v3_report["metadata"]["selected_metrics"]["confusion_matrix"]

print("Current-risk Random Forest confusion matrix")
display(
    pd.DataFrame(
        risk_matrix,
        index=[f"Actual: {x}" for x in risk_labels],
        columns=[f"Predicted: {x}" for x in risk_labels],
    )
)
print("Overall-order-risk v3 Random Forest confusion matrix")
display(
    pd.DataFrame(
        order_matrix,
        index=["Actual: Low", "Actual: High"],
        columns=["Predicted: Low", "Predicted: High"],
    )
)

## 5. Calibrated three-day early-warning evidence

The selected model for each subtype receives nested order-grouped sigmoid calibration. A target-specific threshold is then selected from grouped out-of-fold calibrated probabilities by maximising Macro-F1. A universal 50% threshold is not assumed.

In [ ]:
early_report = json.loads(
    (
        PROJECT_ROOT / "reports" / "component3_early_warning_step5b_calibrated" / "evaluation.json"
    ).read_text()
)
early_rows = []
for _target_name, target in early_report["targets"].items():
    calibrated = target["calibration"]["nested_threshold_validation_metrics"]
    early_rows.append(
        {
            "Target": target["display_name"],
            "Selected model": target["selected_model"],
            "Accuracy": calibrated["accuracy"],
            "Macro-F1": calibrated["macro_f1"],
            "Positive F1": calibrated["f1"],
            "Threshold": target["calibration"]["decision_threshold"],
        }
    )
early_results = pd.DataFrame(early_rows)
display(
    early_results.style.format(
        {"Accuracy": "{:.2%}", "Macro-F1": "{:.2%}", "Positive F1": "{:.2%}", "Threshold": "{:.2%}"}
    )
)

## 6. Hyperparameter optimization and explicit ensemble comparison

The panel feedback is addressed with a nested grouped experiment:

- **Outer validation:** Leave-One-Group-Out by `Bulk_Order_ID`.
- **Inner optimization:** three-fold GroupKFold with GridSearchCV.
- **Tuned models:** class-balanced Logistic Regression and Random Forest.
- **Explicit ensemble:** probability-based Soft Voting between the tuned models.
- **Primary metric:** Macro-F1.

An outer test order is absent from parameter selection, ensemble-weight selection, and fitting. The study creates evidence and does not overwrite active artifacts automatically.

In [ ]:
optimization_dir = PROJECT_ROOT / "reports" / "component3_model_optimization"
optimization_report = json.loads((optimization_dir / "evaluation.json").read_text())
optimization_results = pd.read_csv(optimization_dir / "model_comparison.csv")
display(
    optimization_results.style.format({"Accuracy": "{:.2%}", "Macro_F1": "{:.2%}", "F1": "{:.2%}"})
)

print("Search spaces:")
display(
    pd.DataFrame(
        [
            {"Model": name, "Search space": json.dumps(values)}
            for name, values in optimization_report["search_spaces"].items()
        ]
    )
)

In [ ]:
selected_parameter_rows = []
for _target_name, target in optimization_report["targets"].items():
    for candidate_name, candidate in target["candidates"].items():
        if candidate["selected"]:
            selected_parameter_rows.append(
                {
                    "Target": target["display_name"],
                    "Nested winner": candidate_name,
                    "Accuracy": candidate["accuracy"],
                    "Macro-F1": candidate["macro_f1"],
                    "Positive F1": candidate["f1"],
                    "Full-data grouped-search parameters": json.dumps(
                        candidate["final_full_data_best_parameters"], sort_keys=True
                    ),
                }
            )
selected_parameters = pd.DataFrame(selected_parameter_rows)
display(
    selected_parameters.style.format(
        {"Accuracy": "{:.2%}", "Macro-F1": "{:.2%}", "Positive F1": "{:.2%}"}
    )
)

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("Optional chart skipped. Install matplotlib to display it.")

if plt is not None:
    chart_data = optimization_results.pivot(
        index="Display_Name", columns="Candidate", values="Macro_F1"
    )
    axis = chart_data.plot(kind="bar", figsize=(12, 5), ylim=(0, 1), rot=15)
    axis.set_title("Nested unseen-order Macro-F1: tuned models and Soft Voting")
    axis.set_xlabel("Early-warning target")
    axis.set_ylabel("Macro-F1")
    axis.legend(title="Candidate", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## 7. Parameter explanation

| Parameter | Values considered / setting | Why |
|---|---|---|
| Logistic Regression `C` | 0.05, 0.2, 1.0, 5.0 | Controls regularisation; smaller values reduce overfitting |
| `class_weight` | balanced | Reduces majority-class dominance |
| `max_iter` | 5000 | Ensures Logistic Regression convergence |
| Random Forest `n_estimators` | 200, 400 | Compares ensemble stability and compute cost |
| Random Forest `max_depth` | None, 8 | Compares unrestricted and constrained tree complexity |
| `min_samples_leaf` | 1, 3 | Tests fine patterns against stronger overfitting control |
| `max_features` | sqrt | Encourages diversity among trees |
| Soft-voting weights | 1:1, 2:1, 1:2 | Tests equal and model-emphasised probability combinations |
| `random_state` | 42 | Makes experiments reproducible |

Selected settings: machine warning uses Logistic `C=5.0`, a 400-tree Random Forest, and voting weights `2:1`; quality uses Logistic `C=0.05`; output/schedule uses Logistic `C=1.0`.

## 8. How to answer the panel comments

### Why were the models selected?

> Models were selected using predictions for complete held-out bulk orders. Macro-F1 was prioritised because the emergency targets were imbalanced. Random Forest was selected where nonlinear feature interactions improved balanced performance, while Logistic Regression was selected for targets where the smaller regularised model generalised better.

### What parameters were used, and why?

> Logistic Regression regularisation, Random Forest tree count/depth/leaf size, and Soft Voting weights were optimized using inner grouped GridSearchCV. Bulk Order ID was the grouping boundary, so an outer test order could not influence its own parameter selection.

### Was an ensemble evaluated?

> Yes. Random Forest, Extra Trees, Gradient Boosting and XGBoost were already ensemble candidates. A separate tuned Soft Voting ensemble combining Logistic Regression and Random Forest was also evaluated. It achieved the best nested Macro-F1 for machine-breakdown warning.

### Why were all optimized models not deployed automatically?

> Complexity alone is not a reason for deployment. An optimized model is promoted only if comparable held-out evidence improves, probability calibration remains valid, and integration tests pass. The output/schedule tuning result did not exceed the existing calibrated evidence, so the working artifact was retained.

## 9. Validity and limitations

- The real historical evidence contains a limited number of independent bulk orders.
- Model 2 has limited independent real Low Risk order coverage, so its near-perfect row-level score is not treated as production proof.
- Worker shortage is used in current-day detection, but a separate three-day worker-shortage warning artifact is not available.
- Early-warning thresholds and probabilities are calibrated using grouped research evidence, but broader prospective validation is still required.
- The recovery engine is deterministic and explainable; it is intentionally not presented as an ML prediction.

**Conclusion:** Component 3 is a functioning academic prototype with reproducible model comparison, group-safe validation, calibrated warnings, hyperparameter optimization, explicit ensemble evidence, and an explainable recovery workflow.

## 10. Optional: reproduce the optimization report

The following command regenerates the JSON, CSV, and Markdown optimization evidence without replacing active model artifacts:

```bash
python train_component3_optimized.py
```

For the presentation, use the already generated reports unless the panel specifically asks for a live retraining demonstration.